# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Mục tiêu:** nâng độ chính xác chuỗi biển 2 dòng (hiện ~0,60 trên tập đánh giá 2.801 mẫu) bằng cách
fine-tune model recognition trên đúng phân phối production (strip 2-dòng-ghép-ngang, cao 64 px).

**Chuẩn bị trước ở máy local** (đã làm sẵn trong repo):
```
backend/.venv/Scripts/python scripts/dataset/build_rec_finetune_set.py
cd datasets/processed && zip -r rec_finetune.zip rec_finetune
```
Tải `rec_finetune.zip` (~54 MB, 6.672 train + 571 val) lên Google Drive của bạn, thư mục `DATN/`.

**Chạy trên Colab GPU** (Runtime → Change runtime type → T4).


In [ ]:
# 1) Kiem tra GPU
!nvidia-smi -L


In [ ]:
# 2a) Cai dat PaddlePaddle GPU 3.0 — CHI co tren index cua Paddle (PyPI dung o 2.6.2)
#     Timeout la chuyen thuong cua CDN nay: --retries 8 de pip tu thu lai
!pip install -q --timeout 300 --retries 8 paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 2b) DU PHONG neu 2a van timeout: tai wheel bang wget -c (noi lai duoc khi dut mang),
#     chay lai cell nay bao nhieu lan cung duoc cho toi khi tai xong
import re, urllib.request
html = urllib.request.urlopen('https://www.paddlepaddle.org.cn/packages/stable/cu118/paddlepaddle-gpu/').read().decode()
name = re.search(r'paddlepaddle_gpu-3\.0\.0[^"]*cp312[^"]*linux_x86_64\.whl', html).group(0)
url = f'https://paddle-whl.bj.bcebos.com/stable/cu118/paddlepaddle-gpu/{name}'
print(url)
!wget -c -q --show-progress --tries=10 --timeout=60 {url}
!pip install -q paddlepaddle_gpu-3.0.0*.whl
import paddle; print('CUDA:', paddle.device.is_compiled_with_cuda())


In [ ]:
# 2c) Clone PaddleOCR va cai requirements (tach rieng de de chan doan)
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -q --timeout 180 --retries 5 -r requirements.txt


In [ ]:
# 3) Mount Drive va giai nen dataset
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/DATN/rec_finetune.zip -d /content/data
!wc -l /content/data/rec_finetune/train.txt /content/data/rec_finetune/val.txt
!head -2 /content/data/rec_finetune/train.txt


In [ ]:
# 4) Tai pretrained weights cua en_PP-OCRv5_mobile_rec
#    NEU link doi, tra cuu bang model list trong docs cua PaddleOCR (PP-OCRv5).
!mkdir -p /content/pretrained
!wget -q -O /content/pretrained/en_PP-OCRv5_mobile_rec_pretrained.pdparams \
    https://paddleocr.bj.bcebos.com/PP-OCRv5/english/en_PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la /content/pretrained


In [ ]:
# 5) Tim config goc cua model
import glob
candidates = glob.glob('configs/rec/**/*en_PP-OCRv5_mobile*.yml', recursive=True)
print(candidates)
CONFIG = candidates[0]


In [ ]:
# 6) Huan luyen (fine-tune)
#    - charset 36 ky tu (quyet dinh Phase 1) da nam trong dict36.txt cua dataset
#    - max_text_length=10 (bien VN dai nhat 9 ky tu + du phong)
#    - epoch/lr la diem khoi dau hop ly; theo doi acc tren val de dieu chinh
!python tools/train.py -c {CONFIG} \
  -o Global.pretrained_model=/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Global.epoch_num=30 \
     Global.save_epoch_step=5 \
     Global.eval_batch_step='[0,200]' \
     Global.save_model_dir=/content/output/rec_vn \
     Optimizer.lr.learning_rate=0.0001 \
     Train.dataset.data_dir=/content/data/rec_finetune \
     Train.dataset.label_file_list='[/content/data/rec_finetune/train.txt]' \
     Train.loader.batch_size_per_card=128 \
     Eval.dataset.data_dir=/content/data/rec_finetune \
     Eval.dataset.label_file_list='[/content/data/rec_finetune/val.txt]'


In [ ]:
# 7) Danh gia checkpoint tot nhat
!python tools/eval.py -c {CONFIG} \
  -o Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Eval.dataset.data_dir=/content/data/rec_finetune \
     Eval.dataset.label_file_list='[/content/data/rec_finetune/val.txt]'


In [ ]:
# 8) Export inference model va luu ve Drive
!python tools/export_model.py -c {CONFIG} \
  -o Global.checkpoints=/content/output/rec_vn/best_accuracy \
     Global.character_dict_path=/content/data/rec_finetune/dict36.txt \
     Global.use_space_char=False \
     Global.max_text_length=10 \
     Global.save_inference_dir=/content/output/rec_vn_inference
!cd /content/output && zip -r rec_vn_inference.zip rec_vn_inference
!cp /content/output/rec_vn_inference.zip /content/drive/MyDrive/DATN/


## Tích hợp về hệ thống

1. Giải nén `rec_vn_inference.zip` vào `models/rec_finetuned/` của repo.
2. Bật qua biến môi trường (compose đã nối sẵn):
```
ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned   # trong .env, rồi docker compose up -d
```
   Chạy local: `set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned`.
3. **Đo lại theo đúng kỷ luật** (không có bước này thì chưa được công bố số):
   - `ai/evaluation/ocr_accuracy.py` — A4–A7 toàn tập, so với baseline trong `docs/reports/16-*`;
   - bộ hồi quy: 16 ảnh lõi (`expected.json`), 13 ca rescue, demo video;
   - nếu A6 hai dòng không tăng ≥ 5 điểm thì coi như thất bại, gỡ cờ và ghi lại trung thực.
